# Python Data Wrangling with `pandas`
* * *
<div class="alert alert-success">  

### Learning Objectives

By the end of this workshop you will be able to:
* Explain what a DataFrame is and how to inspect one
* Select specific rows and columns using `.loc` and boolean masks
* Handle missing values, merge tables, and group data for aggregation
</div>

### Icons Used in This Notebook
📚 **Concept**: The core idea — what it is and why it matters, independent of pandas syntax.<br>
🔔 **Question**: A quick question to check understanding.<br>
🥊 **Challenge**: Interactive exercise.<br>
💡 **Tip**: A more efficient or effective way to do something.<br>
⚠️ **Warning**: Tricky stuff or common mistakes.<br>
🤖 **AI Note**: How this concept relates to using AI tools effectively.<br>

### Sections
1. [The `DataFrame` Object](#dataframe)
2. [Indexing Data](#indexing)
3. [Boolean Indexing](#boolean)
4. [Missing Data](#missing)
5. [Merging DataFrames](#merging)
6. [Grouping and Aggregating](#grouping)
7. [Further Learning](#further)

<div class="alert alert-info">

### 🤖 AI Tools and This Workshop

AI assistants (Claude, ChatGPT, Copilot) can write pandas syntax on demand. So this workshop deliberately focuses on **concepts over syntax**: the mental models that let you know *what to ask for*, *whether the output is correct*, and *why something broke*.

Every 📚 **Concept** section covers something you need to understand regardless of what tool generates the code.

</div>

<a id='dataframe'></a>
# 1. The `DataFrame` Object

<div class="alert alert-warning">

### 📚 Concept: What is a DataFrame?

A **DataFrame** is a two-dimensional table: rows are individual observations (e.g. one country in one year), and columns are variables (e.g. unemployment rate, date, country code).

Two things make a DataFrame different from a plain spreadsheet:

1. **Every column has a fixed data type** (number, text, date, boolean). Operations on a column are only valid if the type supports them — you can take the mean of a number column, but not of a text column. Checking types is always the first step with unfamiliar data.

2. **Every row has an index label** (by default just 0, 1, 2...). The index is how pandas identifies rows. It matters when you start merging and slicing data.

Anything that operates on a single column returns a **Series** — a one-dimensional labelled array. Understanding the DataFrame/Series distinction prevents a lot of confusing errors.

</div>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
unemployment = pd.read_csv('../data/country_total.csv')
unemployment.head()

Four attributes that answer the first questions you should always ask about a new DataFrame:

In [ ]:
unemployment.shape     # how many rows and columns?

In [ ]:
unemployment.columns   # what are the variables?

In [ ]:
unemployment.dtypes    # what type is each column?

In [ ]:
unemployment.describe() # what is the range of numeric values?

⚠️ **Warning**: `.describe()` silently skips non-numeric columns. If a column that should be numeric shows up as `object` dtype, that is a signal the data needs cleaning (stray characters, mixed types, etc.).

💡 **Tip**: The [pandas documentation](http://pandas.pydata.org/pandas-docs/stable/) is the authoritative reference. AI assistants can explain functions quickly but occasionally hallucinate argument names — verify against the docs when something seems off.

## 🥊 Challenge 1: Exploring a New DataFrame

Load the countries dataset below and answer:
1. What columns does it contain, and what are their types?
2. How many rows and columns does it have?
3. What are the min and max values of the numeric columns?

<details><summary><a>Click for hint</a></summary>
Use <code>.columns</code>, <code>.dtypes</code>, <code>.shape</code>, and <code>.describe()</code>.
</details>

In [ ]:
countries_url = 'https://raw.githubusercontent.com/dlab-berkeley/Python-Data-Wrangling/main/data/countries.csv'
countries = pd.read_csv(countries_url)
countries.head()

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

<a id='indexing'></a>
# 2. Indexing Data

<div class="alert alert-warning">

### 📚 Concept: What is indexing?

**Indexing** means selecting a specific subset of a DataFrame — particular rows, particular columns, or both.

The key mental model: pandas indexing always works in **[rows, columns]** order. Think of it like specifying a cell range in a spreadsheet, but with the power to name exactly what you want.

There are two main approaches:
- **Label-based** (`.loc`): you refer to rows and columns by their *names*. This is what you should use by default.
- **Position-based** (`.iloc`): you refer to rows and columns by their *integer position* (0, 1, 2...). Useful when you don't know column names in advance.

The result type depends on what you select: a single column gives a **Series**, multiple columns give a **DataFrame**. This distinction matters when you chain further operations.

</div>

Here is how rows and columns map onto the `countries` DataFrame:

<img src="../images/df_diagram.svg" align="left" width="500" alt="diagram of pandas dataframe">

## `.loc` in practice

The syntax is: `df.loc[row_selection, column_selection]`
Use `:` to mean 'all' in either position.

In [ ]:
# A single value — row 3, column 'google_country_code'
countries.loc[3, 'google_country_code']

In [ ]:
# A range of rows, all columns
countries.loc[:4, :]

In [ ]:
# A range of rows, one column -> returns a Series
countries.loc[2:4, 'name_en']

In [ ]:
# A range of rows, a list of columns -> returns a DataFrame
countries.loc[19:29, ['name_en', 'longitude']]

🔔 **Question**: Run `type(countries.loc[2:4, 'name_en'])` and `type(countries.loc[2:4, ['name_en']])`. What is the difference? Why does it matter?

In [ ]:
print(type(countries.loc[2:4, 'name_en']))       # single column name -> Series
print(type(countries.loc[2:4, ['name_en']]))      # list of column names -> DataFrame

## 🥊 Challenge 2: Indexing with `.loc`

Select rows 10 through 20 of `countries`, then compute their average `latitude`.

<details>
    <summary><a>Click for hint</a></summary>
    <code>countries.loc[10:20, 'latitude'].mean()</code>
</details>

In [ ]:
# YOUR CODE HERE

<a id='boolean'></a>
# 3. Boolean Indexing

<div class="alert alert-warning">

### 📚 Concept: What is a boolean mask?

Sometimes you don't know *which* row numbers you want — you want all rows that satisfy a **condition**. For example: all countries with unemployment above 10%, or all observations from Spain.

This works through a **boolean mask**: a True/False array with one entry per row. Rows where the condition is True are kept; rows where it is False are dropped.

The three-step pattern:
1. **Write the condition** on a column: `df['col'] == 'value'` → produces a boolean mask
2. **Pass the mask back** into the DataFrame: `df[mask]` → returns only the True rows

If you have used SQL, this is exactly a `WHERE` clause. If you have used Excel, it is the same as a filter.

</div>

Let's make the mask visible by building it up one step at a time:

In [ ]:
# A small slice to keep output readable
test = countries.loc[20:25, :]
test

In [ ]:
# Step 1: write the condition -> produces a True/False mask
test['country_group'] == 'non-eu'

In [ ]:
# Step 2: pass the mask into the DataFrame -> keeps only True rows
test[test['country_group'] == 'non-eu']

Any comparison operator works: `==`, `!=`, `>`, `<`, `>=`, `<=`:

In [ ]:
# All countries with longitude greater than 25
countries[countries['longitude'] > 25]

## Combining Conditions

You can chain conditions with `&` (AND) and `|` (OR). **Always wrap each condition in parentheses** — pandas requires this because of Python operator precedence:

```python
df[(condition_1) & (condition_2)]   # both must be True
df[(condition_1) | (condition_2)]   # either must be True
```

In [ ]:
# Countries with longitude between 25 and 30
countries[(countries['longitude'] > 25) & (countries['longitude'] < 30)]

In [ ]:
# Countries with longitude > 30 OR < 0
countries[(countries['longitude'] > 30) | (countries['longitude'] < 0)]

## 🥊 Challenge 3: Boolean Indexing

1. Compute the average longitude of all countries and save it to `average_long`
2. Use a boolean mask to filter to only the countries with above-average longitude

<details>
    <summary><a>Click for hint</a></summary>
    <code>average_long = countries['longitude'].mean()</code><br>
    Then: <code>countries[countries['longitude'] > average_long]</code>
</details>

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

🤖 **AI Note**: Boolean indexing is one of the places where AI-generated code looks correct but isn't. A missing parenthesis around a condition, or using `and` instead of `&`, will either error or silently return wrong results. Knowing the three-step pattern (condition → mask → filter) helps you read and verify any filtering code, regardless of who wrote it.

<a id='missing'></a>
# 4. Missing Data

<div class="alert alert-warning">

### 📚 Concept: What is missing data, and why does it matter?

In pandas, missing values are represented as `NaN` (Not a Number). This is a distinct state — it is not the same as `0`, `False`, or an empty string.

Missing data **propagates**: `NaN + 5 = NaN`. This means a single missing value can silently corrupt aggregations (means, sums) if you do not handle it deliberately.

There are three strategies, and the right one depends on your context:
- **Drop**: remove rows (or columns) with missing values. Simple, but loses data.
- **Impute**: fill in a substitute value — the column mean, median, a forward-fill from the previous row, etc. Preserves row count, but introduces assumptions.
- **Leave as-is**: many pandas and statistical functions have `skipna=True` by default, so they handle NaNs gracefully. Sometimes no action is needed.

Always ask *why* data is missing before deciding how to handle it.

</div>

In [ ]:
unemployment = pd.read_csv('../data/cleaned_country_totals.csv')
unemployment['date'] = pd.to_datetime(unemployment['date'])
unemployment.head()

`.isna()` returns a boolean mask — True where values are missing. Since `True == 1` in Python, chaining `.sum()` counts missing values per column:

In [ ]:
unemployment.isna().sum()

Only `unemployment_rate` has missing values. Here we'll drop those rows:

In [ ]:
unemployment = unemployment.dropna(subset=['unemployment_rate'])
unemployment.isna().sum()  # verify

💡 **Tip**: `subset` takes a **list** of column names — even for a single column, use `['col']`. `.dropna()` returns a copy; assigning it back is what makes the change stick.

<a id='merging'></a>
# 5. Merging DataFrames

<div class="alert alert-warning">

### 📚 Concept: What is a join?

Data is often split across multiple tables to avoid redundancy. A **join** combines two tables into one by matching rows on a **shared key column**.

The critical question is: what happens to rows that have no match in the other table?

| Join type | Keeps |
|---|---|
| **Inner** (default) | Only rows with a match in *both* tables |
| **Left** | All rows from the left table; NaN where there is no match on the right |
| **Right** | All rows from the right table; NaN where there is no match on the left |
| **Outer** | All rows from both tables; NaN where there is no match on either side |

If you have used SQL, this is a `JOIN`. If you have used Excel, it is a `VLOOKUP` — but cleaner.

A good diagnostic: compare `.shape` before and after the merge. Unexpected row loss usually means a key mismatch (whitespace, capitalisation, encoding).

</div>

Our `unemployment` DataFrame has country codes but no full names. `countries` has both. The shared key is the `country` column:

In [ ]:
unemployment.head(2)

In [ ]:
countries.head(2)

In [ ]:
print('unemployment rows:', unemployment.shape[0])
print('countries rows:    ', countries.shape[0])

In [ ]:
unemployment_merged = pd.merge(unemployment, countries, on='country')
print('merged rows:', unemployment_merged.shape[0])
unemployment_merged.head()

🔔 **Question**: The merged DataFrame has fewer rows than `unemployment` alone. Why?

🤖 **AI Note**: AI tools will usually write `pd.merge()` correctly when asked. What they commonly get wrong is the join type. If you just say 'merge these two tables', you will get an inner join — which silently drops non-matching rows. Knowing the join types means you can specify exactly what you need.

<a id='grouping'></a>
# 6. Grouping and Aggregating Data

<div class="alert alert-warning">

### 📚 Concept: What is split-apply-combine?

Grouping is one of the most fundamental patterns in data analysis. The underlying logic is called **split-apply-combine**:

1. **Split**: divide the data into groups based on the unique values of a column (e.g. one group per country)
2. **Apply**: compute a statistic within each group independently (e.g. the maximum unemployment rate for each country)
3. **Combine**: collect those per-group results into a new table

You already know this pattern under other names:
- **Excel**: a pivot table
- **SQL**: `GROUP BY` with an aggregate function (`COUNT`, `AVG`, `MAX`, ...)
- **Everyday language**: 'What is the average salary *by department*?' — the *by* signals a groupby

The key thing AI tools get wrong here: grouping on the wrong column, or aggregating the wrong variable. Understanding the three steps means you can catch those errors immediately.

</div>

## Quick grouping: `.value_counts()`

For the simplest case — counting how many rows belong to each group — `.value_counts()` is a one-liner:

<img src="../images/valcounts.svg" align="left" width="400" alt="diagram of value_counts">

In [ ]:
unemployment_merged['name_en'].value_counts()

## 🥊 Challenge 4: Value Counts

Use `.value_counts()` to find how many observations are from EU vs. non-EU countries.

<details><summary><a>Click for hint</a></summary>
Which column distinguishes EU from non-EU? Check <code>unemployment_merged.columns</code>.
</details>

In [ ]:
# YOUR CODE HERE

## General grouping: `.groupby()`

For any aggregation beyond counting — means, sums, maxima — use `.groupby()`. The pattern maps directly to the split-apply-combine steps:

```python
df.groupby('split_column')['apply_column'].combine_function()
#          ^ Step 1: split  ^ Step 2: apply  ^ Step 3: combine
```

<img src="../images/groupby.svg" align="left" width="500" alt="diagram of groupby">

In [ ]:
# Split by country_group, apply .mean() to unemployment_rate, combine
unemployment_merged.groupby('country_group')['unemployment_rate'].mean()

The split-apply-combine result should match manually filtering to one group and computing the same statistic. This is a useful sanity check:

In [ ]:
# Manual check: filter to EU rows only, then take the mean
eu_mask = unemployment_merged['country_group'] == 'eu'
unemployment_merged.loc[eu_mask, 'unemployment_rate'].mean()

Common aggregate functions: `.mean()`, `.sum()`, `.max()`, `.min()`, `.count()`, `.median()`, `.std()`

🤖 **AI Note**: This is the area where AI-generated code most often has subtle bugs. Typical issues: grouping on the wrong column, using `.count()` when `.sum()` was intended, or aggregating a column that contains NaNs without realising it. The split-apply-combine mental model is your checklist for reviewing any groupby output.

## 🥊 Challenge 5: Groupby

Use `.groupby()` to find the **maximum** unemployment rate ever recorded for each country, sorted from largest to smallest.

Before writing any code: identify the three steps.
- **Split** on which column?
- **Apply** to which column?
- **Combine** using which function?

<details><summary><a>Click for hint</a></summary>
<code>unemployment_merged.groupby('name_en')['unemployment_rate'].max().sort_values(ascending=False)</code>
</details>

In [ ]:
# YOUR CODE HERE

<a id='further'></a>
# 7. Further Learning

This section is for after the workshop. It is organised around a simple question: **what is worth learning that AI tools cannot shortcut for you?**

## 7.1 Practice on Real Data

The fastest way to solidify today's concepts is to use them on data you actually care about. A good workflow: pick a dataset, write a question in plain English, ask an AI to generate the pandas code, then *verify* the output using what you learned today.

**Suggested datasets:**

| Dataset | Why it's good for practice |
|---|---|
| [Our World in Data](https://ourworldindata.org/charts) | Download any chart as CSV. Rich time-series data — good for groupby and merge practice. |
| [Kaggle: Titanic](https://www.kaggle.com/c/titanic/data) | The standard beginner dataset. Has missing values, categorical columns, and natural grouping questions. |
| [TidyTuesday](https://github.com/rfordatascience/tidytuesday) | A new clean dataset every week with a clear topic. Huge variety. |
| [EU Open Data Portal](https://data.europa.eu/en) | More Eurostat data — extends the datasets from this workshop. |

**Practice questions to try on the Eurostat data from this workshop:**
1. Which country had the highest unemployment in 2010? In 2020? Did the ranking change?
2. How many distinct countries are in the dataset? How many years of data does each one have?
3. Filter to a single country and look at how its unemployment rate changed over time (describe it in words — you do not need a chart).

## 7.2 Concepts Worth Understanding Deeply

These are areas where deeper knowledge makes you better at data work *and* better at using AI for data work — because you can write clearer prompts and catch wrong output faster.

---

### SQL

If you understand SQL `GROUP BY`, `JOIN`, and `WHERE`, pandas will feel obvious — because pandas is essentially SQL for Python. SQL is also the language of databases, which is where most real-world data lives before it reaches a CSV.

A useful exercise: take any pandas operation from today and ask an AI to rewrite it as SQL. Compare the two. The concepts map directly.

Good starting point: [SQLZoo](https://sqlzoo.net/) — free, interactive, browser-based.

---

### Tidy Data

Hadley Wickham's [tidy data](https://vita.had.co.nz/papers/tidy-data.pdf) principles describe what makes a dataset *well-structured*:
- Each variable is a column
- Each observation is a row
- Each type of observational unit is a separate table

Most messy data problems — columns that should be rows, multiple variables in one column, inconsistent naming — are violations of these three rules. Recognising violations is a prerequisite for fixing them (whether you write the code or ask AI to).

---

### Statistical Thinking

AI can compute a mean. It cannot tell you whether a mean is the right summary for your data.

A few ideas worth knowing:
- **Simpson's paradox**: a trend can appear in subgroups but reverse when you aggregate them. Groupby results can mislead if you are not controlling for confounding variables.
- **Mean vs median**: the mean is pulled by outliers; the median is not. For skewed data (income, prices, counts), the median is often more informative.
- **Sample size**: a max unemployment rate for a country with 3 observations is much less reliable than one with 300. `.value_counts()` before aggregating is a good habit.

A short, free read: [Calling Bullshit: Data Reasoning in a Digital World](https://www.callingbullshit.org/) by Carl Bergstrom and Jevin West.

---

### Data Wrangling Patterns: reshape, pivot, melt

Today covered the core operations. The next tier includes:
- **`pd.melt()`** — convert wide-format tables (one column per year) to long format (one row per year). Required before most groupby operations on time-series data.
- **`pd.pivot_table()`** — the inverse: go from long to wide. Equivalent to an Excel pivot table.
- **`df.str` accessor** — string operations on text columns (split, extract, replace). Essential for cleaning messy text fields.

These come up constantly in real data work. AI can write them, but you need to know they exist and what they do to ask for them.

## 7.3 Tools Worth Knowing

---

### ydata-profiling (automated EDA)

Generates a full exploratory data analysis report — distributions, correlations, missing values, duplicates — with one line of code:

```python
from ydata_profiling import ProfileReport
ProfileReport(df).to_notebook_iframe()
```

Useful as a first step before any analysis. Install with `pip install ydata-profiling`.

---

### DuckDB (SQL on DataFrames)

DuckDB lets you run SQL directly on pandas DataFrames — fast, in-memory, no server needed:

```python
import duckdb
duckdb.sql('SELECT country, MAX(unemployment_rate) FROM unemployment GROUP BY country')
```

If you find yourself more comfortable writing SQL than pandas, DuckDB is a legitimate alternative for many wrangling tasks. It also handles files larger than memory. Install with `pip install duckdb`.

---

### pandera (data validation)

One underrated skill: catching when your data is *different from what you assumed*. `pandera` lets you write explicit expectations about your data and raises an error if they are violated:

```python
import pandera as pa
schema = pa.DataFrameSchema({
    'unemployment_rate': pa.Column(float, pa.Check.between(0, 100)),
    'country': pa.Column(str),
})
schema.validate(df)  # raises if data doesn't match
```

This is especially valuable when your data pipeline is AI-assisted — AI-generated transformations can silently produce values outside expected ranges. Install with `pip install pandera`.

---

### Polars (faster pandas)

Polars is a pandas alternative written in Rust. It is significantly faster on large datasets and has a cleaner API for chaining operations. The concepts from today transfer directly (filter, groupby, join all exist). Worth knowing it exists if you start hitting pandas performance limits. Install with `pip install polars`.

## 7.4 Using AI Effectively for Data Work

AI tools are genuinely useful for data wrangling. Here are the habits that make the difference between using them well and getting plausible-looking wrong answers.

---

### Write prompts that include your data's structure

Vague: *'how do I group data in pandas?'*

Useful:
> I have a DataFrame called `unemployment_merged` with columns: `country` (str), `date` (datetime), `unemployment_rate` (float), `country_group` (str, values 'eu'/'non-eu'). I want to find the maximum unemployment rate per country, sorted descending.

Pasting `df.head()` output and `df.dtypes` into your prompt is often the fastest way to get correct, runnable code.

---

### Always verify against a known subset

Before applying an AI-generated transformation to a full dataset, test it on a small slice where you can manually check the result. If the output matches your expectation on 5 rows, it is much more likely to be correct on 5,000.

```python
# Test on a small, checkable slice first
sample = df[df['country'] == 'ES'].head(10)
# ... apply transformation ...
# inspect sample output manually
```

---

### Ask AI to explain its code, not just write it

After getting code, follow up with: *'explain each step of this code in plain English.'* If the explanation reveals the code is doing something other than what you intended, you have caught a bug before it affected your results. This also accelerates your own learning.

---

### Ask AI to generate sanity checks

After writing a data transformation, ask: *'what assertions could I add to verify this transformation is correct?'* AI is good at generating edge cases and checks that you might not think of yourself:

```python
# Example AI-suggested checks after a merge:
assert unemployment_merged.shape[0] > 0, 'Merge produced empty result'
assert unemployment_merged['unemployment_rate'].notna().all(), 'Unexpected NaNs after merge'
assert set(unemployment_merged['country_group'].unique()) == {'eu', 'non-eu'}
```

# 🎉 Well Done!

Here are the six concepts this workshop covered, and why each one matters independent of syntax:

| Concept | The mental model |
|---|---|
| **DataFrame** | A table with typed columns and a row index |
| **Indexing** | Select subsets by label: `[rows, columns]` |
| **Boolean mask** | A True/False filter — same as SQL WHERE, Excel filter |
| **Missing data** | NaN is a distinct value; choose to drop, impute, or tolerate |
| **Join** | Combine tables on a shared key; the join type controls what rows survive |
| **Split-apply-combine** | Group → aggregate → collect; the foundation of pivot tables and GROUP BY |

## Next Steps

- **Data Visualization** — the table from Challenge 5 is a natural bar chart. Learn to build charts in D-Lab's [Python Data Visualization workshop](https://github.com/dlab-berkeley/Python-Data-Visualization).
- **Python Intermediate** — [D-Lab workshop](https://github.com/dlab-berkeley/Python-Intermediate-Pilot)
- **pandas documentation** — [pandas.pydata.org](https://pandas.pydata.org/pandas-docs/stable/)